Assignment:
1. Load daily SPX prices
2. Identify tail-risk events (monthly)
3. Simulate Wheel strategy on single stock

In [57]:
import pandas as pd
from os.path import join
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import math

### Load SPX daily prices

In [58]:
# load daily SPX prices
spx_path = join("../../", "market_data", "daily", "SPX.csv")
spx = pd.read_csv(spx_path, parse_dates=["Date"]) \
    .sort_values("Date") \
    .set_index("Date")

# monthly return approximated as 21-trading-day return
spx["ret_1M"] = spx["Close"].pct_change(21)

### Identify Tail-Risk events

In [59]:
threshold = 0.05
spx["is_tail_event"] = spx["ret_1M"] < -threshold

tail_events = spx.loc[spx["is_tail_event"], ["Close", "ret_1M"]].copy()

print(f"Threshold: {-threshold:.2%}")
print(f"Tail-risk event count: {tail_events.shape[0]}")
print(tail_events.tail(10))

Threshold: -5.00%
Tail-risk event count: 26
              Close    ret_1M
Date                         
2025-04-04  5074.08 -0.115786
2025-04-07  5062.25 -0.122691
2025-04-08  4982.77 -0.112527
2025-04-10  5268.05 -0.059159
2025-04-16  5275.70 -0.060371
2025-04-17  5282.70 -0.069175
2025-04-21  5158.20 -0.089122
2025-04-22  5287.76 -0.067013
2025-04-23  5375.86 -0.067916
2025-04-24  5484.77 -0.050528


In [60]:
# plot SPX price and highlight tail-risk events
fig = px.line(spx, y="Close", title="SPX Price")
fig.add_scatter(x=tail_events.index, y=tail_events["Close"], mode="markers", marker=dict(color="red", size=7), name="Tail-Risk Events")
fig.update_layout(xaxis_title=None, yaxis_title=None)
fig.show()

### Simulate the Wheel strategy

In [61]:
# load daily prices for MSFT
msft_path = join("../../", "market_data", "daily", "MSFT.csv")
msft = pd.read_csv(msft_path, parse_dates=["Date"]) \
    .sort_values("Date") \
    .set_index("Date")

In [62]:
# dummy vol parameters for 1W options
vol = 0.25
r = 0.02
T = 7 / 365       # 1W option
strike_pct_put = 0.02 # % from spot
strike_pct_call = 0.01 # % from spot
contract_size = 100

In [63]:
def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_call(S: float, K: float, T: float, r: float, sigma: float) -> float:
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)


def bs_put(S: float, K: float, T: float, r: float, sigma: float) -> float:
    if T <= 0 or sigma <= 0:
        return max(K - S, 0.0)
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)

In [64]:
weekly_close = msft["Close"].resample("W-FRI").last().dropna()

# start with enough cash to secure 1 put assignment (100 shares)
initial_cash = contract_size * weekly_close.iloc[0]
cash = float(initial_cash)
shares = 0    # initial number of shares

rows = []
for i in range(len(weekly_close) - 1):
    start_date = weekly_close.index[i]
    expiry_date = weekly_close.index[i + 1]
    S0 = float(weekly_close.iloc[i])
    S1 = float(weekly_close.iloc[i + 1])

    if shares == 0:
        # flat position - sell cash-secured put
        K_put = S0 * (1.0 - strike_pct_put)
        prem = bs_put(S0, K_put, T, r, vol)
        cash += prem * contract_size

        assigned = S1 < K_put
        if assigned:
            cash -= K_put * contract_size
            shares = contract_size

        action = "sell_put"
        strike = K_put
        exercised = assigned
    else:
        # sell covered call
        K_call = S0 * (1.0 + strike_pct_call)
        prem = bs_call(S0, K_call, T, r, vol)
        cash += prem * contract_size

        called_away = S1 > K_call
        if called_away:
            cash += K_call * contract_size
            shares = 0

        action = "sell_call"
        strike = K_call
        exercised = called_away

    portfolio_value = cash + shares * S1
    rows.append(
        {
            "start_date": start_date,
            "expiry_date": expiry_date,
            "spot_start": S0,
            "spot_expiry": S1,
            "action": action,
            "strike": strike,
            "premium_per_share": prem,
            "premium_total": prem * contract_size,
            "exercised": exercised,
            "shares_end": shares,
            "cash_end": cash,
            "portfolio_value": portfolio_value,
        }
    )

wheel_df = pd.DataFrame(rows).set_index("expiry_date")

print(f"Initial capital: {initial_cash:,.2f}")
print(f"Final portfolio value: {wheel_df['portfolio_value'].iloc[-1]:,.2f}")
print(f"Total return: {wheel_df['portfolio_value'].iloc[-1] / initial_cash - 1:.2%}")
print(f"Put assignments: {((wheel_df['action'] == 'sell_put') & (wheel_df['exercised'])).sum()}")
print(f"Call exercises: {((wheel_df['action'] == 'sell_call') & (wheel_df['exercised'])).sum()}")

fig = px.line(wheel_df, y="portfolio_value", title="Wheel Strategy Equity Curve (MSFT)")
fig.update_layout(xaxis_title=None, yaxis_title="Portfolio Value")
fig.show()

wheel_df.tail(12)

Initial capital: 41,929.29
Final portfolio value: 43,998.65
Total return: 4.94%
Put assignments: 8
Call exercises: 8


,start_date,spot_start,spot_expiry,action,strike,premium_per_share,premium_total,exercised,shares_end,cash_end,portfolio_value
expiry_date,,,,,,,,,,,
2025-12-26,2025-12-19,484.813446,486.599365,sell_put,475.117177,2.826546,282.654616,False,0,46538.157762,46538.157762
2026-01-02,2025-12-26,486.599365,471.863007,sell_put,476.867378,2.836958,283.695838,True,100,-864.884193,46321.416466
2026-01-09,2026-01-02,471.863007,478.188568,sell_call,476.581637,4.528774,452.877403,True,0,47246.156876,47246.156876
2026-01-16,2026-01-09,478.188568,458.812775,sell_put,468.624797,2.787922,278.792198,True,100,662.469399,46543.746865
2026-01-23,2026-01-16,458.812775,464.888916,sell_call,463.400902,4.403523,440.352253,True,0,47442.911892,47442.911892
2026-01-30,2026-01-23,464.888916,429.310120,sell_put,455.591138,2.710383,271.038271,True,100,2154.836394,45085.848356
2026-02-06,2026-01-30,429.310120,400.226501,sell_call,433.603221,4.120366,412.036649,False,100,2566.873043,42589.523190
2026-02-13,2026-02-06,400.226501,400.406097,sell_call,404.228766,3.841232,384.123223,False,100,2950.996266,42991.606007
2026-02-20,2026-02-13,400.406097,397.230011,sell_call,404.410158,3.842956,384.295593,False,100,3335.291858,43058.292957
